In [1]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [3]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [4]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Topic Subject

### CQ4.01u - Which SOSA sensors publish to which MQTT Topics?

In [5]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?sensor ?topic WHERE {
  ?sensor a sosa:Sensor ;
          mqtt:observesTopic ?topic .
  ?topic a mqtt:Topic .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,sensor,topic
0,ex:Sensor_Sensor_37b,ex:Topic_sensor


### CQ4.02u - Which SOSA actuators listen to which MQTT Topics?

In [6]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?actuator ?topic WHERE {
  ?actuator a sosa:Actuator ;
            mqtt:listensToTopic ?topic .
  ?topic a mqtt:Topic .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,actuator,topic
0,ex:Actuator_Actuator1,ex:Topic_actuator


### CQ4.03u - Which SOSA FeaturesOfInterest, Properties etc. appear as MQTT TopicSubjects and on which MQTT Topics?

In [7]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?topic ?subject ?type WHERE {
  ?topic a mqtt:Topic ;
         mqtt:hasSubject ?subject .
  ?subject a ?type .
  FILTER( STRSTARTS(STR(?type), STR(sosa:)) )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,topic,subject,type
0,ex:Topic_sensor,ex:FeatureOfInterest_RFID_87A3F1,sosaFeatureOfInterest
1,ex:Topic_sensor,ex:FeatureOfInterest_RFID_87A3F2,sosaFeatureOfInterest
2,ex:Topic_sensor,ex:FeatureOfInterest_RFID_87A3F3,sosaFeatureOfInterest
3,ex:Topic_actuator,ex:FeatureOfInterest_RFID_87A3F1,sosaFeatureOfInterest
4,ex:Topic_actuator,ex:FeatureOfInterest_RFID_87A3F2,sosaFeatureOfInterest
5,ex:Topic_actuator,ex:FeatureOfInterest_RFID_87A3F3,sosaFeatureOfInterest


### CQ4.04u - Which MQTT Topics are stored?

In [8]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?topic WHERE {
  ?topic a mqtt:Topic ;
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,topic
0,ex:Topic_sensor
1,ex:Topic_actuator


### CQ4.05u - Which MQTT filter patterns (with wildcards) are stored?

In [9]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?filter ?pattern WHERE {
  ?filter a mqtt:TopicFilter ;
          mqtt:hasFilterPattern ?pattern .
  FILTER(CONTAINS(?pattern, "#") || CONTAINS(?pattern, "+"))
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,filter,pattern


### CQ4.06u - Which MQTT Topic Filters match a given MQTT Topic?

In [10]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?filter ?topic WHERE {
  ?filter a mqtt:TopicFilter ;
          mqtt:matchesTopic ?topic .
  ?topic a mqtt:Topic .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,filter,topic


### CQ4.07u - Which MQTT Topics are matched by a given MQTT Topic Filter?

In [11]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?topic ?filter WHERE {
  ?topic a mqtt:Topic ;
  OPTIONAL { ?topic mqtt:isMatchedByFilter ?filter . ?filter a mqtt:TopicFilter }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,topic,filter
0,ex:Topic_sensor,NaN
1,ex:Topic_actuator,NaN
